# MedRoute — Demonstração Completa
## Tech Challenge Fase 2 · PosTech IA para Devs · Turma 8IADT · FIAP

Este notebook demonstra o sistema completo de otimização de rotas hospitalares, cobrindo:

1. **Dados e modelagem** — pontos de entrega, veículos e restrições
2. **Algoritmo Genético (VRP)** — execução e visualização de convergência
3. **Mapas interativos** — rotas otimizadas e comparativo antes/depois
4. **Experimentos comparativos** — 3 configurações do AG
5. **LLM** — instruções para motoristas, relatório executivo e Q&A

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath("../.."))

from src.data.mock_data import DELIVERY_POINTS, VEHICLES
from src.data.distances import build_distance_matrix

build_distance_matrix(DELIVERY_POINTS)

print("Pontos de entrega:")
for p in DELIVERY_POINTS:
    tag = " (BASE)" if p.id == 0 else f" | prioridade: {p.priority} | {p.demand} kg"
    print(f"  [{p.id}] {p.name}{tag}")

print(f"\nVeículos:")
for v in VEHICLES:
    print(f"  {v.name}: cap={v.capacity} kg, autonomia={v.max_distance} km")

## 2. Executando o Algoritmo Genético

In [ ]:
from src.genetic_algorithm.ga_adapter import run_genetic_algorithm

result = run_genetic_algorithm(
    points=DELIVERY_POINTS,
    vehicles=VEHICLES,
    population_size=50,
    generations=100,
    mutation_rate=0.1,
    seed=42,
)

improvement = (1 - result.optimized_distance / result.initial_distance) * 100 if result.initial_distance > 0 else 0

print(f"Distância inicial:   {result.initial_distance:.2f} km")
print(f"Distância otimizada: {result.optimized_distance:.2f} km")
print(f"Melhoria:            {improvement:.1f}%")
print()
for r in result.routes:
    stops = " → ".join(s.name for s in r.stops) if r.stops else "— sem paradas —"
    print(f"{r.vehicle.name}: {stops}")
    print(f"  {r.total_distance:.2f} km | {r.total_load:.1f}/{r.vehicle.capacity:.0f} kg")

## 3. Gráficos de Convergência e Eficiência

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Image, display

os.makedirs("../../output", exist_ok=True)

from src.visualization.charts import (
    plot_fitness_convergence,
    plot_distance_comparison,
    plot_vehicle_load,
    plot_route_distances,
)

plot_fitness_convergence(result,    "../../output/convergence.png")
plot_distance_comparison(result,    "../../output/distance_comparison.png")
plot_vehicle_load(result,           "../../output/vehicle_load.png")
plot_route_distances(result,        "../../output/route_distances.png")

for path, title in [
    ("../../output/convergence.png",         "Convergência do AG"),
    ("../../output/distance_comparison.png", "Comparativo de Distância"),
    ("../../output/vehicle_load.png",        "Carga por Veículo"),
    ("../../output/route_distances.png",     "Distância por Veículo"),
]:
    print(f"\n{title}")
    display(Image(filename=path))

## 4. Mapas Interativos

Os mapas são arquivos HTML — abra no navegador para ver a animação completa.

- `output/route_map.html` — rotas otimizadas com AntPath animado, LayerControl, Fullscreen, MiniMap
- `output/comparison_map.html` — DualMap lado a lado: antes vs depois da otimização

In [ ]:
from src.visualization.route_map import create_route_map
from src.visualization.comparison_map import create_comparison_map

create_route_map(result, DELIVERY_POINTS, output_path="../../output/route_map.html")
create_comparison_map(result, DELIVERY_POINTS, output_path="../../output/comparison_map.html")

print("Mapas gerados:")
print("  output/route_map.html       — abra no navegador")
print("  output/comparison_map.html  — abra no navegador")

## 5. Experimentos Comparativos

Compara 3 configurações distintas do AG para determinar a melhor abordagem.

In [ ]:
from src.experiments.main import run_experiments
from src.results.main import print_comparison_table, plot_convergence_comparison, plot_improvement_bar

exp_results = run_experiments("../../output/experiments")

print_comparison_table(exp_results)
plot_convergence_comparison(exp_results, "../../output/experiments")
plot_improvement_bar(exp_results, "../../output/experiments")

for path, title in [
    ("../../output/experiments/convergence_comparison.png", "Convergência — 3 Experimentos"),
    ("../../output/experiments/improvement_comparison.png", "Melhoria por Configuração"),
]:
    print(f"\n{title}")
    display(Image(filename=path))

## 6. Integração com LLM

Gera instruções para motoristas, relatório executivo e responde perguntas sobre as rotas.

> **Modo OpenAI**: defina `OPENAI_API_KEY` no ambiente para usar GPT-4o-mini.  
> **Modo offline**: sem a chave, usa templates estruturados equivalentes.

In [ ]:
from src.llm.main import generate_driver_instructions, generate_route_report, ask_question

print("=" * 60)
print("INSTRUÇÕES PARA MOTORISTAS")
print("=" * 60)
instructions = generate_driver_instructions(result, DELIVERY_POINTS)
print(instructions)

print("\n" + "=" * 60)
print("RELATÓRIO EXECUTIVO")
print("=" * 60)
report = generate_route_report(result, exp_results)
print(report)

In [ ]:
print("=" * 60)
print("Q&A — Perguntas sobre as Rotas")
print("=" * 60)

questions = [
    "Quais são os pontos com entrega crítica?",
    "Qual a distância total otimizada?",
    "Como estão distribuídas as entregas por veículo?",
    "Quais são as sugestões para melhorar as rotas?",
]

for q in questions:
    print(f"\nPergunta: {q}")
    print(f"Resposta: {ask_question(q, result, DELIVERY_POINTS)}")